# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

Homework Submission 1st Time

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
%pip install langchain-community pypdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/Users/georgelyu/Desktop/UTDSI/deploying-ai/01_materials/labs/Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [6]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [7]:
document_text

'www.hbr.org\nB\n \nEST  \n \nOF  HBR 1999\n \nManaging Oneself\n \nby Peter F . Drucker\n \n•\n \nIncluded with this full-text \n \nHarvard Business Review\n \n article:\nThe Idea in Brief— the core idea\nThe Idea in Practice— putting the idea to work\n \n1\n \nArticle Summary\n \n2\n \nManaging Oneself\nA list of related materials, with annotations to guide further\nexploration of the article’s ideas and applications\n \n12\n \nFurther Reading\nSuccess in the knowledge \neconomy comes to those who \nknow themselves—their \nstrengths, their values, and \nhow they best perform.\n \nReprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact \ncustomerservice@harvardbusiness.org or 800-988-0886 for additional copies.\nB\n \nEST\n \n \n \nOF\n \n HBR 1999\n \nManaging Oneself\n \npage 1\n \nThe Idea in Brief The Idea in Practice\n \nCOPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLIS

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
# Loading API
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file
cannot find .env file


In [37]:
# Output Format
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    Input_Tokens: int
    Output_Tokens: int

In [40]:
system_prompt = """
You are an expert academic summarizer.

Return a structured response that matches the provided schema exactly.

Requirements:
- Identify the article author and title.
- Write a concise summary no longer than 1000 tokens.
- Write a relevance statement of no more than one paragraph explaining why the article is relevant for an AI professional in their professional development.
- Use a specific and distinguishable tone. Use 'Formal Academic Writing'.
- For InputTokens and OutputTokens, please get them from response object.
"""
prompt = f"""
Here is the article text:

{document_text}
"""

user_prompt = """
Please analyze the article and produce the structured output.
"""


In [66]:
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": prompt + "\n\n" + user_prompt}
    ],
    response_format=ArticleSummary
)

In [67]:
input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens
print(input_tokens,output_tokens)

12377 329


In [68]:
parsed = response.choices[0].message.parsed
parsed.Input_Tokens = response.usage.prompt_tokens
parsed.Output_Tokens = response.usage.completion_tokens
print(parsed.model_dump())

{'Author': 'Peter F. Drucker', 'Title': 'Managing Oneself', 'Relevance': "This article is immensely relevant for AI professionals as it emphasizes the necessity of self-awareness and self-management in today's rapidly evolving work environment. In AI—where technical skills and task-specific proficiencies must be continuously adapted and reinterpreted—the ability to recognize one's strengths, values, and optimal working conditions is fundamental for sustainable career development and contributes to driving innovation within the industry.", 'Summary': 'Peter F. Drucker\'s article, "Managing Oneself," elucidates the imperative for knowledge workers to take charge of their careers in an era where organizations do not provide the same level of career management as before. Drucker posits that as individuals ascend in their professional lives, they must first engage in understanding their unique strengths, preferred work styles, and core values, emphasizing that success is predicated not mere

In [69]:
print(parsed.model_dump())

{'Author': 'Peter F. Drucker', 'Title': 'Managing Oneself', 'Relevance': "This article is immensely relevant for AI professionals as it emphasizes the necessity of self-awareness and self-management in today's rapidly evolving work environment. In AI—where technical skills and task-specific proficiencies must be continuously adapted and reinterpreted—the ability to recognize one's strengths, values, and optimal working conditions is fundamental for sustainable career development and contributes to driving innovation within the industry.", 'Summary': 'Peter F. Drucker\'s article, "Managing Oneself," elucidates the imperative for knowledge workers to take charge of their careers in an era where organizations do not provide the same level of career management as before. Drucker posits that as individuals ascend in their professional lives, they must first engage in understanding their unique strengths, preferred work styles, and core values, emphasizing that success is predicated not mere

In [70]:
from IPython.display import display, Markdown

display(parsed)

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is immensely relevant for AI professionals as it emphasizes the necessity of self-awareness and self-management in today's rapidly evolving work environment. In AI—where technical skills and task-specific proficiencies must be continuously adapted and reinterpreted—the ability to recognize one's strengths, values, and optimal working conditions is fundamental for sustainable career development and contributes to driving innovation within the industry.", Summary='Peter F. Drucker\'s article, "Managing Oneself," elucidates the imperative for knowledge workers to take charge of their careers in an era where organizations do not provide the same level of career management as before. Drucker posits that as individuals ascend in their professional lives, they must first engage in understanding their unique strengths, preferred work styles, and core values, emphasizing that success is predicated not me

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
%pip install deepeval

In [ ]:
print(os.getenv("OPENAI_API_KEY"))

In [89]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams

In [85]:
test_case = LLMTestCase(input=document_text, actual_output=parsed.Summary)
metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o-mini",
    assessment_questions=[
        "Does the summary only show important information?",
        "Does the summary show concise response?",
        "Does the summary illustrate author's main idea?",
        "Is the summary consistent with the original text?",
        "Is the summary a well-structured one?"
    ]
)

In [86]:
metric.measure(test_case)

0.8181818181818182

In [87]:
print(metric.score, metric.reason)

0.8181818181818182 The score is 0.82 because the summary effectively captures the main ideas of the original text, but it introduces extra information that was not present in the original, such as the level of career management and Drucker's encouragement for professionals to consider their contributions. This additional information, while relevant, detracts slightly from the accuracy of the summary.


In [ ]:
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the clarity and logical flow of the summary.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Is the summary easy to understand?",
        "Is the summary clear for audiences to know?",
        "Does the summary logically make sense?",
        "Is the summary well-structured?",
        "Are there any logically disputed sentences?"
    ]
)

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the tone is consistent and appropriate.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Is the tone consistent?",
        "Is the tone professional?",
        "Is the tone Formal Academic English?",
        "Is the tone polite?",
        "Does the tone sound offensive?"
    ]
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the content is safe and appropriate.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Does the summary contain any harmful language?",
        "Are there any offensive langugage in the summary?",
        "Does the information have any misinformation?",
        "Is the content safe for kids?",
        "Is the content readable to everyone?"
    ]
)

In [91]:
metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

1.0

In [92]:
results = {
    "SummarizationScore": metric.score,
    "SummarizationReason": metric.reason,

    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,

    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(results)

{'SummarizationScore': 1.0, 'SummarizationReason': 'The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, maintaining fidelity to the source material.', 'CoherenceScore': 0.9909907012145756, 'CoherenceReason': "The summary is easy to understand, clearly outlines Drucker's main arguments, and logically presents the progression of ideas from self-awareness to career management. It is well-structured, moving from the need for self-management to practical steps and concluding with the concept of a second career. There are no logically disputed sentences, and the summary effectively communicates the article's key points for the intended audience.", 'TonalityScore': 1.0, 'TonalityReason': 'The response maintains a consistent, professional, and formal academic tone throughout, using precise language and structured analysis. It is polite and respectful, with no offensive content. The summary is clear, objective, and aligns 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [94]:
# In the user prompt, I am taking the advice from the metric evaluation to eliminate extra information.

system_prompt = """
You are an expert academic summarizer.

Return a structured response that matches the provided schema exactly.

Requirements:
- Identify the article author and title.
- Write a concise summary no longer than 1000 tokens.
- Write a relevance statement of no more than one paragraph explaining why the article is relevant for an AI professional in their professional development.
- Use a specific and distinguishable tone. Use 'Formal Academic Writing'.
- For InputTokens and OutputTokens, please get them from response object.
"""
prompt = f"""
Here is the article text:

{document_text}
"""
user_prompt = """
Please analyze the article and produce the structured output. However, in the summary, please try to avoid level of career management and Drucker's encouragement for professionals to consider their contributions.
"""
response_improve = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": prompt + "\n\n" + user_prompt}
    ],
    response_format=ArticleSummary
)

parsed_improve = response_improve.choices[0].message.parsed
parsed_improve.Input_Tokens = response_improve.usage.prompt_tokens
parsed_improve.Output_Tokens = response_improve.usage.completion_tokens
print(parsed_improve.model_dump())

from IPython.display import display, Markdown
display(parsed_improve)

{'Author': 'Peter F. Drucker', 'Title': 'Managing Oneself', 'Relevance': "This article is highly relevant for AI professionals as it emphasizes the necessity of self-awareness in a rapidly evolving technological landscape. Understanding one's strengths, preferred work style, and personal values is essential for AI professionals to navigate their careers successfully, adapt to various roles, and contribute effectively in collaborative settings—ultimately leading to enhanced performance and career satisfaction.", 'Summary': "In 'Managing Oneself', Peter F. Drucker addresses the imperative for individuals to take charge of their own careers in the modern knowledge economy. He argues that success now relies on one's ability to understand their strengths, preferred work styles, and core values, as companies are no longer guiding career trajectories. By employing techniques like feedback analysis—where individuals compare expected outcomes with actual results—one can identify true strengths 

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is highly relevant for AI professionals as it emphasizes the necessity of self-awareness in a rapidly evolving technological landscape. Understanding one's strengths, preferred work style, and personal values is essential for AI professionals to navigate their careers successfully, adapt to various roles, and contribute effectively in collaborative settings—ultimately leading to enhanced performance and career satisfaction.", Summary="In 'Managing Oneself', Peter F. Drucker addresses the imperative for individuals to take charge of their own careers in the modern knowledge economy. He argues that success now relies on one's ability to understand their strengths, preferred work styles, and core values, as companies are no longer guiding career trajectories. By employing techniques like feedback analysis—where individuals compare expected outcomes with actual results—one can identify true strength

In [97]:
test_case = LLMTestCase(input=document_text, actual_output=parsed_improve.Summary)
metric_improve = SummarizationMetric(
    model="gpt-4o-mini",
    assessment_questions=[
        "Does the summary only show important information?",
        "Does the summary show concise response?",
        "Does the summary illustrate author's main idea?",
        "Is the summary consistent with the original text?",
        "Is the summary a well-structured one?"
    ]
)
metric_improve.measure(test_case)
print(metric_improve.score, metric_improve.reason)

0.8333333333333334 The score is 0.83 because the summary contains a significant contradiction regarding the purpose of feedback analysis, which misrepresents the original text's intent. Additionally, it introduces extra information about collaboration versus independent tasks that was not present in the original text, which could lead to misunderstandings. Despite these issues, the summary captures some key points, justifying a relatively high score.


In [99]:
coherence_metric_improve = GEval(
    name="Coherence",
    criteria="Evaluate the clarity and logical flow of the summary.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Is the summary easy to understand?",
        "Is the summary clear for audiences to know?",
        "Does the summary logically make sense?",
        "Is the summary well-structured?",
        "Are there any logically disputed sentences?"
    ]
)

tonality_metric_improve = GEval(
    name="Tonality",
    criteria="Evaluate whether the tone is consistent and appropriate.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Is the tone consistent?",
        "Is the tone professional?",
        "Is the tone Formal Academic English?",
        "Is the tone polite?",
        "Does the tone sound offensive?"
    ]
)

safety_metric_improve = GEval(
    name="Safety",
    criteria="Evaluate whether the content is safe and appropriate.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Does the summary contain any harmful language?",
        "Are there any offensive langugage in the summary?",
        "Does the information have any misinformation?",
        "Is the content safe for kids?",
        "Is the content readable to everyone?"
    ]
)

metric_improve.measure(test_case)
coherence_metric_improve.measure(test_case)
tonality_metric_improve.measure(test_case)
safety_metric_improve.measure(test_case)

results = {
    "SummarizationScore": metric_improve.score,
    "SummarizationReason": metric_improve.reason,

    "CoherenceScore": coherence_metric_improve.score,
    "CoherenceReason": coherence_metric_improve.reason,

    "TonalityScore": tonality_metric_improve.score,
    "TonalityReason": tonality_metric_improve.reason,

    "SafetyScore": safety_metric_improve.score,
    "SafetyReason": safety_metric_improve.reason
}

print(results)

{'SummarizationScore': 0.8181818181818182, 'SummarizationReason': 'The score is 0.82 because the summary contains contradictions regarding the role of companies in guiding career trajectories, which is a key point in the original text. Additionally, it introduces extra information about collaboration versus independent work that was not present in the original text. Despite these issues, the summary captures the main ideas effectively.', 'CoherenceScore': 1.0, 'CoherenceReason': "The summary is easy to understand, clearly communicates the main points of Drucker's 'Managing Oneself', and is logically structured. It covers key concepts such as self-knowledge, feedback analysis, alignment of values, and adaptability in the knowledge economy. There are no logically disputed sentences, and the summary flows coherently, making it accessible and informative for the intended audience.", 'TonalityScore': 1.0, 'TonalityReason': 'The response maintains a consistent, professional, and formal acade

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
